# SEBAL Notebook 5: Refined Fluxes and Stability Correction  
**Scene:** Landsat 7 (2020-01-19)  
**Region:** Mississippi Delta (EPSG:32615)

This notebook continues the SEBAL workflow from the first-pass energy balance results.  
It is used to apply stability correction, refine sensible heat flux and latent heat flux, and update evapotranspiration estimates for the study area.

## Starting Point — First-Pass SEBAL Results

The first-pass SEBAL energy balance was completed earlier.  
The following rasters already exist in `04_indices` and will be used here:

• H (first-pass sensible heat flux)  
• Rn (net radiation)  
• G (soil heat flux)  
• ETinst (first-pass instantaneous ET)  
• ET24 (first-pass daily ET)

This notebook continues with stability correction and refined flux computation.

## 1. Import Required Python Libraries and Utility Functions

In [1]:
import os
import rasterio
import numpy as np
from Utility import get_file_dataframe
from Utility import load_rasters
from Utility import calculate_NDVI
from Utility import calculate_albedo
from Utility import extract_mask_mean
from Utility import read_raster
from Utility import write_raster
from Utility import check_raster

## 2. Set SEBAL working directory

In [2]:
os.chdir("..")
os.getcwd()

'G:\\Collaborations\\Mentee\\UF_Anitha Madapakula\\Scripts\\Python\\SEBAL_20200119_MSDelta'

## 3. Define directories and input

In [4]:
indices_dir = "04_indices"
met_dir     = "03_processed_met"

date = "20200119"
hour = "16Z"
region = "MSDelta"

## 4. Load temperature difference raster (dT)

In [ ]:
dT_path = os.path.join(indices_dir, f"dT_{date}_{hour}_{region}.tif")

dT, _, _ = read_raster(dT_path)

print("dT shape  :", dT.shape)
print("dT min/max:",
      float(dT[dT > -9999].min()),
      float(dT[dT > -9999].max()))

## 8. Refined Sensible Heat H
The refined H using:
$$ 𝐻 = \rho 𝐶_𝑝 \frac{𝑑𝑇}{𝑟_{𝑎ℎ}}
$$
where: $\rho = .225 kg/m^3$ and $𝐶_𝑝 = 1004 J/kg/K$

In [ ]:
rho = 1.225
cp  = 1004

H_refined = np.full(dT.shape, -9999.0, dtype="float32")

valid_H = (dT > -9999) & (rah > 0)

H_refined[valid_H] = rho * cp * dT[valid_H] / rah[valid_H]

print("H_refined min/max:",
      float(H_refined[H_refined > -9999].min()),
      float(H_refined[H_refined > -9999].max()))

# Save refined sensible heat raster

Href_out = os.path.join(indices_dir, f"H_refined_{date}_{hour}_{region}.tif")
write_raster(Href_out, profile, H_refined)

## 9. Monin–Obukhov Length L
$$
L = - \frac{\rho C_p T_a u_*^3}{k g H}
$$

In [ ]:
g = 9.81

L = np.full(H_refined.shape, -9999.0, dtype="float32")

valid_L = (H_refined != 0) & (ustar > 0) & (TA > -9999)

L[valid_L] = - (rho * cp * TA[valid_L] * (ustar[valid_L]**3)) / (k * g * H_refined[valid_L])

print("L min/max:",
      float(L[L > -9999].min()),
      float(L[L > -9999].max()))

## 10. Stability correction functions ($psi_m$, $psi_h$)
$$
x = (1 - 16 \frac {z_2}{L})^0.25
$$
$$
psi_m = ln(\frac{1+x}{2}) + ln(\frac{1+x^2}{2}) - 2 tan^{-1}x  + \frac{\pi}{2}
$$
$$
psi_h = 2ln(\frac{1+x^2}{2})
$$


In [ ]:
psi_m = np.zeros(L.shape, dtype="float32")
psi_h = np.zeros(L.shape, dtype="float32")

# heights
z2 = 2.0
z1 = 0.1

# --- unstable (L < 0)
unstable = (L < 0) & (L > -9999)

x = (1 - 16 * (z2 / L[unstable]))**0.25

psi_m[unstable] = (2*np.log((1+x)/2) + np.log((1+x**2)/2) - 2*np.arctan(x)  + np.pi/2)

psi_h[unstable] = 2*np.log((1+x**2)/2)

# --- stable (L > 0)
stable = (L > 0)

psi_m[stable] = -5 * (z2 / L[stable])
psi_h[stable] = -5 * (z2 / L[stable])

print("psi_m range:", psi_m.min(), psi_m.max())
print("psi_h range:", psi_h.min(), psi_h.max())

### 10.1 Stabilize L near zero before using psi

In [ ]:
L_safe = L.copy()

small_pos = (L_safe > 0) & (L_safe < 0.1)
small_neg = (L_safe < 0) & (L_safe > -0.1)

L_safe[small_pos] = 0.1
L_safe[small_neg] = -0.1

print("L_safe min/max:",
      float(L_safe[L_safe > -9999].min()),
      float(L_safe[L_safe > -9999].max()))
unstable = (L_safe < 0) & (L_safe > -9999)
x = (1 - 16 * (z2 / L_safe[unstable]))**0.25

psi_m[unstable] = (
    2*np.log((1+x)/2)
    + np.log((1+x**2)/2)
    - 2*np.arctan(x)
    + np.pi/2
)

psi_h[unstable] = 2*np.log((1+x**2)/2)

# --- stable (L > 0)
stable = (L_safe > 0)
psi_m[stable] = -5 * (z2 / L_safe[stable])
psi_h[stable] = -5 * (z2 / L_safe[stable])
print("psi_m range:", psi_m.min(), psi_m.max())
print("psi_h range:", psi_h.min(), psi_h.max())

## 11. Compute stability-corrected aerodynamic resistance

In [ ]:
rah_corr = np.full(rah.shape, -9999.0, dtype="float32")

valid_corr = (ustar > 0)

rah_corr[valid_corr] = (np.log(z2 / z1) - psi_h[valid_corr]) / (k * ustar[valid_corr])

print("rah_corr min/max:",
      float(rah_corr[rah_corr > -9999].min()),
      float(rah_corr[rah_corr > -9999].max()))

### Fix nonphysical values before saving

In [ ]:
rah_corr_fixed = rah_corr.copy()

rah_corr_fixed[rah_corr_fixed <= 0] = -9999

print("rah_corr min/max:",
      float(rah_corr_fixed[rah_corr_fixed > -9999].min()),
      float(rah_corr_fixed[rah_corr_fixed > -9999].max()))

rahcorr_out = os.path.join(indices_dir, f"rah_corr_{date}_{hour}_{region}.tif")

write_raster(rahcorr_out, profile, rah_corr_fixed, ndata=-9999.0)

## 12. Final refined sensible heat using corrected rah

In [ ]:
H_final = np.full(dT.shape, -9999.0, dtype="float32")

valid_Hf = (dT > -9999) & (rah_corr_fixed > 0)

H_final[valid_Hf] = rho * cp * dT[valid_Hf] / rah_corr_fixed[valid_Hf]

print("H_final min/max:",
      float(H_final[H_final > -9999].min()),
      float(H_final[H_final > -9999].max()))

H_out = os.path.join(indices_dir, f"H_refined_{date}_{hour}_{region}.tif")

write_raster(H_out, profile, H_final, ndata=-9999.0)


### 12.1 Apply lower bound to corrected aerodynamic resistance (rah)

In [ ]:
# ---------------------------------------------------------
# Apply physical lower bound to corrected aerodynamic resistance
# Ensures rah does not approach zero and inflate sensible heat.
# A minimum of 1 s/m is used following standard SEBAL practice.
# ---------------------------------------------------------
rah_corr_bounded = rah_corr_fixed.copy()

rah_min = 1.0   # safe lower bound in s/m
rah_corr_bounded[(rah_corr_bounded > 0) & (rah_corr_bounded < rah_min)] = rah_min

print("rah_corr_bounded min/max:",
      float(rah_corr_bounded[rah_corr_bounded > -9999].min()),
      float(rah_corr_bounded[rah_corr_bounded > -9999].max()))

### 12.2 Load corrected dT raster for refined energy balance

In [ ]:
# ---------------------------------------------------------
# Load corrected dT raster calibrated using corrected rah
# This replaces the earlier first-pass dT in refined H step.
# ---------------------------------------------------------
dTcorr_path = os.path.join(indices_dir, f"dT_corr_{date}_{hour}_{region}.tif")
dT_corr2, _, _ = read_raster(dTcorr_path)

print("dT_corr2 min/max:",
      float(dT_corr2[dT_corr2 > -9999].min()),
      float(dT_corr2[dT_corr2 > -9999].max()))

### 12.3 Compute refined sensible heat using corrected rah and dT

In [ ]:
# ---------------------------------------------------------
# Compute refined sensible heat flux:
# H = rho * cp * dT / rah
# using corrected aerodynamic resistance and corrected dT.
# ---------------------------------------------------------
valid_Hf = (dT_corr2 > -9999) & (rah_corr_bounded > 0)
H_final[valid_Hf] = rho * cp * dT_corr2[valid_Hf] / rah_corr_bounded[valid_Hf]

### 12.4 Sanity check for refined sensible heat

In [ ]:
# ---------------------------------------------------------
# Quick diagnostic check of refined H distribution.
# Used to confirm values fall within physical range.
# ---------------------------------------------------------
# =========================================================
# Check available energy against H_final
# =========================================================

AE = np.full(Rn.shape, -9999.0, dtype="float32")
valid_AE = (Rn > -9999) & (G > -9999)

AE[valid_AE] = Rn[valid_AE] - G[valid_AE]

valid_cmp = (AE > -9999) & (H_final > -9999)

print("AE min/max:", float(AE[valid_cmp].min()), float(AE[valid_cmp].max()))
print("H_final min/max:", float(H_final[valid_cmp].min()), float(H_final[valid_cmp].max()))

print("\nPercentiles:")
for p in [1, 5, 25, 50, 75, 95, 99]:
    print(
        f"{p}% | AE={np.percentile(AE[valid_cmp], p):.3f} "
        f"| H={np.percentile(H_final[valid_cmp], p):.3f}"
    )
print("H_final min/max:",
      float(H_final[H_final > -9999].min()),
      float(H_final[H_final > -9999].max()))

### 12.5 Save corrected final sensible heat raster

In [ ]:
# ---------------------------------------------------------
# Save refined sensible heat raster.
# Uses new filename to avoid overwriting earlier outputs.
# ---------------------------------------------------------
Hfinal_out = os.path.join(indices_dir, f"H_final_{date}_{hour}_{region}.tif")
write_raster(Hfinal_out, profile, H_final, ndata=-9999.0)

## 13. Refined Latent Heat Flux (LE)
### 13.1 Compute refined latent heat flux (LE)

In [ ]:
# ---------------------------------------------------------
# Compute refined latent heat flux using energy balance:
# LE = Rn - G - H_final
# ---------------------------------------------------------

LE_final = np.full(Rn.shape, -9999.0, dtype="float32")

valid_LE = (Rn > -9999) & (G > -9999) & (H_final > -9999)

LE_final[valid_LE] = Rn[valid_LE] - G[valid_LE] - H_final[valid_LE]

print("LE_final min/max:",
      float(LE_final[LE_final > -9999].min()),
      float(LE_final[LE_final > -9999].max()))

### 13.2 Sanity check for refined LE

In [ ]:
# ---------------------------------------------------------
# Check LE distribution for physical realism
# ---------------------------------------------------------

valid = LE_final > -9999

for p in [1, 5, 25, 50, 75, 95, 99]:
    print(f"{p}% =", np.percentile(LE_final[valid], p))

### 13.3 Clamp LE to physical lower bound

In [ ]:
LE_final_clamped = np.full(Rn.shape, -9999.0, dtype="float32")

valid = LE_final > -9999
LE_final_clamped[valid] = np.maximum(LE_final[valid], 0)

print("Clamped LE min/max:",
      float(LE_final_clamped[valid].min()),
      float(LE_final_clamped[valid].max()))

In [ ]:
LE_out = os.path.join(indices_dir, f"LE_final_{date}_{hour}_{region}.tif")
write_raster(LE_out, profile, LE_final_clamped, ndata=-9999.0)

## 14. Refined evaporative fraction (EF)
### 14.1 Compute refined evaporative fraction (EF)

In [ ]:
# ---------------------------------------------------------
# Compute refined evaporative fraction:
# EF = LE / (Rn - G)
# ---------------------------------------------------------

EF_final = np.full(Rn.shape, -9999.0, dtype="float32")

valid_EF = (LE_final_clamped > -9999) & (AE > 0)

EF_final[valid_EF] = LE_final_clamped[valid_EF] / AE[valid_EF]

print("EF_final min/max:",
      float(EF_final[EF_final > -9999].min()),
      float(EF_final[EF_final > -9999].max()))

### 14.2 Save refined EF raster

In [ ]:
# ---------------------------------------------------------
# Save refined evaporative fraction raster
# ---------------------------------------------------------

EF_out = os.path.join(indices_dir, f"EF_final_{date}_{hour}_{region}.tif")

write_raster(EF_out, profile, EF_final, ndata=-9999.0)

## 15. Instantaneous ET
### 15.1 Convert refined LE to instantaneous ET

In [ ]:
# ---------------------------------------------------------
# Convert LE to instantaneous ET (mm/hr)
# ETinst = LE / lambda
# lambda ≈ 2.45 MJ/kg = 2.45e6 J/kg
# ---------------------------------------------------------

lambda_v = 2.45e6

ET_inst = np.full(Rn.shape, -9999.0, dtype="float32")

valid_ET = LE_final_clamped > -9999
ET_inst[valid_ET] = LE_final_clamped[valid_ET] / lambda_v * 3600

print("ET_inst min/max:",
      float(ET_inst[ET_inst > -9999].min()),
      float(ET_inst[ET_inst > -9999].max()))

### 15.2 Save instantaneous ET raster

In [ ]:
# ---------------------------------------------------------
# Save instantaneous ET raster (mm/hr)
# ---------------------------------------------------------

ETinst_out = os.path.join(indices_dir, f"ETinst_final_{date}_{hour}_{region}.tif")

write_raster(ETinst_out, profile, ET_inst, ndata=-9999.0)

### 15.3 Sanity check for ETinst

In [ ]:
valid = ET_inst > -9999

print("Percentiles:")
for p in [1, 5, 25, 50, 75, 95, 99]:
    print(f"{p}% =", np.percentile(ET_inst[valid], p))

## 16. Daily ET from evaporative fraction
### 16.1 Compute daily ET from evaporative fraction

In [ ]:
# Refine daily ET using ratio of refined to first-pass ETinst

ET24_fp_path = os.path.join(indices_dir, f"ET24_{date}_{hour}_{region}.tif")
ETinst_fp_path = os.path.join(indices_dir, f"ETinst_{date}_{hour}_{region}.tif")

ET24_fp, _, _ = read_raster(ET24_fp_path)
ETinst_fp, _, _ = read_raster(ETinst_fp_path)

ET24_refined = np.full(Rn.shape, -9999.0, dtype="float32")

valid_ratio = (ET24_fp > -9999) & (ETinst_fp > 0) & (ET_inst > -9999)

ET24_refined[valid_ratio] = (
    ET24_fp[valid_ratio] * ET_inst[valid_ratio] / ETinst_fp[valid_ratio]
)

print("ET24_refined min/max:",
      float(ET24_refined[ET24_refined > -9999].min()),
      float(ET24_refined[ET24_refined > -9999].max()))

### 16.2 Sanity check for refined daily ET

In [ ]:
valid = ET24_refined > -9999

print("Percentiles:")
for p in [1, 5, 25, 50, 75, 95, 99]:
    print(f"{p}% =", np.percentile(ET24_refined[valid], p))

### 16.3 Save refined daily ET raster

In [ ]:
ET24ref_out = os.path.join(indices_dir, f"ET24_final_{date}_{hour}_{region}.tif")

write_raster(ET24ref_out, profile, ET24_refined, ndata=-9999.0)

## 17. Final summary statistics (refined SEBAL outputs)

In [ ]:
def stats(name, arr):
    valid = arr > -9999
    print(f"\n{name}")
    print("Min:", float(arr[valid].min()))
    print("Max:", float(arr[valid].max()))
    print("Mean:", float(arr[valid].mean()))
    print("Median:", float(np.median(arr[valid])))

stats("H_final (W/m²)", H_final)
stats("LE_final (W/m²)", LE_final)
stats("EF_final (-)", EF_final)
stats("ETinst (mm/hr)", ET_inst)
stats("ET24_final (mm/day)", ET24_refined)

## 18. Compare first-pass vs refined ET
### 18.1 Compute ET difference (refined − first-pass)

In [ ]:
ET24_fp_path = os.path.join(indices_dir, f"ET24_{date}_{hour}_{region}.tif")
ET24_fp, _, _ = read_raster(ET24_fp_path)

ET24_diff = np.full(Rn.shape, -9999.0, dtype="float32")

valid_diff = (ET24_fp > -9999) & (ET24_refined > -9999)

ET24_diff[valid_diff] = ET24_refined[valid_diff] - ET24_fp[valid_diff]

print("ET24_diff min/max:",
      float(ET24_diff[ET24_diff > -9999].min()),
      float(ET24_diff[ET24_diff > -9999].max()))

### 18.2 Difference statistics

In [ ]:
valid = ET24_diff > -9999

print("Mean difference:", float(ET24_diff[valid].mean()))
print("Median difference:", float(np.median(ET24_diff[valid])))

print("\nPercentiles:")
for p in [1, 5, 25, 50, 75, 95, 99]:
    print(f"{p}% =", np.percentile(ET24_diff[valid], p))

### 18.3 Histogram of ET difference

In [ ]:
# Reload rasters after kernel restart
ET24_fp_path = os.path.join(indices_dir, f"ET24_{date}_{hour}_{region}.tif")
ET24_ref_path = os.path.join(indices_dir, f"ET24_final_{date}_{hour}_{region}.tif")

ET24_fp, _, _ = read_raster(ET24_fp_path)
ET24_refined, _, _ = read_raster(ET24_ref_path)

ET24_diff = np.full(Rn.shape, -9999.0, dtype="float32")

valid_diff = (ET24_fp > -9999) & (ET24_refined > -9999)
ET24_diff[valid_diff] = ET24_refined[valid_diff] - ET24_fp[valid_diff]

### 18.3 Histogram of ET difference

Histogram plotting in JupyterLab was skipped due to kernel memory instability for full-scene raster comparison.  
Difference raster statistics were computed successfully, and visualization can be completed in QGIS using the saved ET difference raster.